# 03 - Rules-Based Data Extraction

**Goal:** Convert raw OCR text into structured data (vendor, date, total, items)

**Strategy:**
1. **Vendor:** First strong text line or largest text at top
2. **Date:** Regex matching for various date formats (YYYY-MM-DD, DD/MM/YYYY, etc.)
3. **Total:** Keyword search (TOTAL, AMOUNT, etc.) + currency parsing
4. **Items:** Clustering text by Y-coordinate rows

> **Why rules?** They are fast, deterministic, and easy to debug. LLMs are a fallback.

## Setup

In [11]:
import json
import re
import pandas as pd
from pathlib import Path
from typing import List, Dict, Optional
from datetime import datetime
from dateutil import parser

# Paths
OUT_DIR = Path("../out")
OCR_JSON_FILES = list(OUT_DIR.glob("ocr_*.json"))

print(f"Found {len(OCR_JSON_FILES)} OCR result files")

Found 1 OCR result files


## Step 1: Load Raw OCR Data

Let's load the JSON output from the previous notebook.

In [12]:
# Load the first available JSON file
if OCR_JSON_FILES:
    json_path = OCR_JSON_FILES[0]
    print(f"Loading: {json_path.name}")
    
    with open(json_path, 'r') as f:
        data = json.load(f)
        
    ocr_results = data['results']
    print(f"Loaded {len(ocr_results)} text lines")
else:
    print("⚠️ No OCR JSON files found! Run notebook 02_ocr.ipynb first.")
    ocr_results = []

Loading: ocr_receipt-ocr-original.json
Loaded 188 text lines


In [13]:
# Helper: display top lines to eyeball vendor/date
for i, line in enumerate(ocr_results[:10]):
    print(f"{i}: {line['text']} (conf: {line['confidence']})")

0: Walmart (conf: 0.998)
1: @ (conf: 0.994)
2: Save money. Live better. (conf: 1.0)
3: 3991 (conf: 1.0)
4: - (conf: 0.947)
5: 330 (conf: 1.0)
6: ∞ (conf: 0.305)
7: 339 (conf: 1.0)
8: ( (conf: 0.879)
9: DIANA EARNEST (conf: 0.997)


## Step 2: Extract Vendor

**Heuristic:** The vendor name is usually:
- At the very top
- High confidence
- Often excluding generic words like "Receipt", "Welcome", etc.

In [14]:
def extract_vendor(lines: List[Dict]) -> str:
    """
    Identify vendor name from top lines.
    Strategy: First line that isn't a generic header.
    """
    IGNORE_WORDS = {'receipt', 'welcome', 'tax invoice', 'copy', 'payment'}
    
    for line in lines[:5]:  # Look at top 5 lines
        text = line['text'].strip()
        clean_text = text.lower()
        
        # Skip empty or very short lines
        if len(text) < 2:
            continue
            
        # Skip lines containing ignore words
        if any(w in clean_text for w in IGNORE_WORDS):
            continue
            
        return text
    
    return "Unknown Vendor"

# Test it
vendor = extract_vendor(ocr_results)
print(f"Vendor: {vendor}")

Vendor: Walmart


## Step 3: Extract Date

**Heuristic:** Regex matching for common formats.
- `YYYY-MM-DD`
- `DD/MM/YYYY`
- `DD-MMM-YYYY`

In [15]:
def extract_date(lines: List[Dict]) -> Optional[str]:
    """
    Extract date using regex matching.
    Returns ISO format date string (YYYY-MM-DD).
    """
    # Common date patterns
    date_patterns = [
        r'\d{4}-\d{2}-\d{2}',       # 2024-01-31
        r'\d{2}/\d{2}/\d{4}',       # 31/01/2024
        r'\d{2}-\d{2}-\d{4}',       # 31-01-2024
        r'\d{2}\s+[A-Za-z]{3}\s+\d{4}', # 31 Jan 2024
    ]
    
    for line in lines:
        text = line['text']
        for pattern in date_patterns:
            match = re.search(pattern, text)
            if match:
                date_str = match.group(0)
                try:
                    # Use dateutil parser to handle various formats
                    dt = parser.parse(date_str, dayfirst=True)
                    return dt.strftime("%Y-%m-%d")
                except ValueError:
                    continue
                    
    return None

# Test it
receipt_date = extract_date(ocr_results)
print(f"Date: {receipt_date}")

Date: None


## Step 4: Extract Total Amount

**Heuristic:** Look for lines containing "Total" keywords, then extract the currency amount.

In [16]:
def extract_total(lines: List[Dict]) -> float:
    """
    Find total amount.
    Strategy: 
    1. Find lines with 'TOTAL', 'AMOUNT DUE', etc.
    2. Extract number from that line or the line immediately following (if right-aligned).
    """
    TOTAL_KEYWORDS = ['total', 'amount due', 'grand total', 'balance']
    
    # Regex for currency amount (e.g., $10.50, 10.50, 1,234.50)
    amount_re = r'[\$£€]?\s*(\d{1,3}(?:,\d{3})*(?:\.\d{2})?)'
    
    candidates = []
    
    for i, line in enumerate(lines):
        text = line['text'].lower()
        
        if any(k in text for k in TOTAL_KEYWORDS):
            # Try to find amount in same line
            matches = re.findall(amount_re, line['text'])
            if matches:
                # Take the last match (usually the final total)
                val = matches[-1].replace(',', '')
                try:
                    return float(val)
                except ValueError:
                    pass
            
            # If not in same line, check next line (common in OCR if spacing is wide)
            if i + 1 < len(lines):
                next_line = lines[i+1]['text']
                # Only if next line looks like a number
                matches = re.findall(amount_re, next_line)
                if matches:
                    val = matches[0].replace(',', '')
                    try:
                        return float(val)
                    except ValueError:
                        pass
                        
    return 0.0

# Test it
total_amount = extract_total(ocr_results)
print(f"Total: {total_amount}")

Total: 93.62


## Step 5: Extract Line Items (Advanced)

**Heuristic:** Group text by Y-coordinate.
Items usually consist of: `Name ....... Price`

We group OCR tokens that share roughly the same `center_y`.

In [17]:
def extract_items(lines: List[Dict], y_tolerance=10) -> List[Dict]:
    """
    Extract line items by grouping tokens into rows.
    """
    # 1. Group by Y-coordinate
    rows = []
    if not lines: 
        return []
    
    current_row = [lines[0]]
    
    for line in lines[1:]:
        prev_y = current_row[-1]['center_y']
        curr_y = line['center_y']
        
        if abs(curr_y - prev_y) <= y_tolerance:
            current_row.append(line)
        else:
            rows.append(current_row)
            current_row = [line]
    rows.append(current_row)
    
    # 2. Process each row to separate Item Name vs Price
    items = []
    amount_re = r'^\s*[\$£€]?\s*(\d{1,3}(?:,\d{3})*(?:\.\d{2})?)\s*$'
    
    for row in rows:
        # Sort by X coordinate (left to right)
        row.sort(key=lambda x: x['bbox'][0])
        
        row_text = " ".join([x['text'] for x in row])
        
        # Simple heuristic: Last token is price, rest is name
        # But check if last token matches number format
        last_token = row[-1]['text']
        match = re.match(amount_re, last_token)
        
        if match:
            price = float(match.group(1).replace(',', ''))
            name = " ".join([x['text'] for x in row[:-1]])
            
            # Filter out rows that are likely totals/subtotals
            if 'total' not in name.lower() and 'tax' not in name.lower() and name.strip():
                items.append({
                    'item_name': name,
                    'price': price
                })
                
    return items

# Test items
items = extract_items(ocr_results)
print(f"Found {len(items)} items:")
for item in items:
    print(f"  - {item['item_name']:<30}  {item['price']}")

Found 19 items:
  - PET TOY 004747571658 1.97       7.0
  - FLOPPY PUPPY 004747514846       1.97
  - SSSUPREME S 070060332153        4.97
  - 2.5 SQUEAK 084699803238         5.92
  - MUNCHY DMBEL 068113108796       3.77
  - DOG TREAT 007119013654          2.92
  - HNYMD SMORES 088491226837 H     3.98
  - FRENCH DRSNG 004132100655 H     1.98
  - 3 ORANGES 001466835001 H 5.47   7.0
  - BABY CARROTS 003338366602 I 1.48  1.0
  - COLLARDS 000000004614KI 1.24    7.0
  - CALZONE 005208362080 C          2.5
  - MM RVW MNT 003399105848         19.77
  - STKOBRLPLABL 001558679414       1.97
  - STKOBRLPLABL 001558679414       1.97
  - STKO SUNFLWR 001558679410       0.97
  - STKO SUNFLWR 001558679410 0.97  2.0
  - VISA TEND                       98.21
  - CHANGE DUE                      0.0


## Analysis of Failures

**Observation:** You might notice errors in the item extraction above.
For example: `PET TOY` might show `7.00` instead of `1.97` if the columns aren't perfectly aligned or if the text is wrapped.

**Why this happens:**
- Simple Y-clustering assumes a straight horizontal line.
- It assumes the last number on the right is always the price.
- It fails when there's an internal code (like `040000000 7.00 G`) on the same line.

**Solution:** This is where a **Local LLM** shines. It understands context ("7.00 is likely a quantity or code, 1.97 is the price").

We will explore this in **Notebook 04**.

## Step 6: Validate & Export CSV

Combine everything and save to CSV.

In [18]:
# Create Receipts DataFrame
receipt_data = [{
    'receipt_id': '001', # Placeholder
    'source_file': json_path.name,
    'vendor': vendor,
    'date': receipt_date,
    'total': total_amount
}]

df_receipts = pd.DataFrame(receipt_data)
print("Receipts Data:")
print(df_receipts)

# Create Items DataFrame
df_items = pd.DataFrame(items)
df_items['receipt_id'] = '001'
print("\nItems Data:")
print(df_items.head())

Receipts Data:
  receipt_id                    source_file   vendor  date  total
0        001  ocr_receipt-ocr-original.json  Walmart  None  93.62

Items Data:
                   item_name  price receipt_id
0  PET TOY 004747571658 1.97   7.00        001
1  FLOPPY PUPPY 004747514846   1.97        001
2   SSSUPREME S 070060332153   4.97        001
3    2.5 SQUEAK 084699803238   5.92        001
4  MUNCHY DMBEL 068113108796   3.77        001


In [19]:
# Export to CSV
df_receipts.to_csv(OUT_DIR / "receipts.csv", index=False)
df_items.to_csv(OUT_DIR / "receipt_items.csv", index=False)

print(f"✅ Saved CSVs to {OUT_DIR}/")

✅ Saved CSVs to ../out/


---
## 🎯 Key Takeaways

1. **Extraction is heuristic-heavy:** No rule works for 100% of receipts.
2. **Regex is powerful:** Great for dates and prices.
3. **Spatial grouping:** Essential for table/line item extraction.
4. **Validation:** Check if sum(items) == total (we'll add this later in the full pipeline).

## 🚀 Next Steps

We identified limitations in rules-based extraction. In **Notebook 04**, we'll use a local LLM to fix these errors.